# NeuroSim — Full End-to-End Pipeline Demo

**GSoC 2026 Project #39 — INCF / EBRAINS**  
*Automating In-Silico Stimulation for Non-Invasive Biomarker Discovery*

**Mentor:** Dr. Khushbu Agarwal | **Author:** Md. Shamsul Alam

---

This notebook is the **integrated demonstration** of all three NeuroSim modules running in sequence:

```
Multi-Site Raw Data
       │
       ▼  Module A
  Blind neuroCombat  ──►  Site-harmonized features
       │
       ▼  Module B-1
  Spectral Inversion ──►  Schur-stable directed A matrix
       │
       ▼  Module B-2 / C
  Gramian + Energy   ──►  Transition energy landscape
       │
       ▼  Module C
  Modal Controllability ►  Facilitator node ranking
```

---

> **Dr. Agarwal's requirements addressed:**
> 1. ✅ Blind ComBat (fit on HC, apply to clinical — disease signal preserved)
> 2. ✅ Schur-stable A matrix (spectral radius < 1 guaranteed by design)
> 3. ✅ Facilitator node detection (Modal Controllability ranked output)
> 4. ✅ Full reproducibility (fixed seeds throughout)


## 0. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.decomposition import PCA

# ── NeuroSim modules ──────────────────────────────────────────────────────────
from neurosim.harmonization import fit_combat, apply_combat, blind_harmonize
from neurosim.connectivity import (
    spectral_inversion_solver,
    check_schur_stability,
    normalize_matrix,
)
from neurosim.control.gramian import compute_gramian
from neurosim.control.energy import minimum_energy
from neurosim.control.metrics import modal_controllability, average_controllability, rank_facilitator_nodes

# Reproducible RNG
RNG = np.random.default_rng(seed=2026)

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold'})

print("✓ All NeuroSim modules imported successfully.")
print("✓ Reproducible seed: 2026")

---
## Module A — Blind neuroCombat Harmonization

**Design rationale:** We fit ComBat **exclusively on Healthy Controls (HCP)**. The clinical cohorts
(ADNI-AD, AUD, Epilepsy) are harmonized using these locked parameters — their pathological variance
is never used to estimate scanner effects, preserving disease biomarker signal.

In [ ]:
# ── Experimental configuration ────────────────────────────────────────────────
N_PARCELS   = 100   # Schaefer-100 parcellation
N_HC        = 120   # Healthy Control subjects (HCP)
N_CLINICAL  = 60    # Clinical subjects per cohort
SCANNER_OFFSET = 3.0  # Simulated site B scanner bias

# ── Simulate HCP reference cohort (2 scanners) ────────────────────────────────
hc_site_A = RNG.standard_normal((N_PARCELS, N_HC // 2))                     # HCP 3T
hc_site_B = RNG.standard_normal((N_PARCELS, N_HC // 2)) + SCANNER_OFFSET    # HCP 7T
hc_data   = np.hstack([hc_site_A, hc_site_B])
hc_labels = ['HCP_3T'] * (N_HC // 2) + ['HCP_7T'] * (N_HC // 2)

# ── Simulate 3 clinical cohorts (all from HCP_3T scanner) ────────────────────
adni_data     = RNG.standard_normal((N_PARCELS, N_CLINICAL)) + 0.6   # AD: mild mean shift
aud_data      = RNG.standard_normal((N_PARCELS, N_CLINICAL)) - 0.4   # AUD: negative shift
epilepsy_data = RNG.standard_normal((N_PARCELS, N_CLINICAL)) + 1.2   # Epilepsy: large shift
clinical_labels = ['HCP_3T'] * N_CLINICAL

print(f"HC cohort shape:       {hc_data.shape}  (2 scanners, {N_HC} subjects)")
print(f"Clinical cohort shape: {adni_data.shape} (per group)")
print(f"\nSite A mean: {hc_site_A.mean():.3f}")
print(f"Site B mean: {hc_site_B.mean():.3f}  ← scanner offset of {SCANNER_OFFSET}")

In [ ]:
# ── Step A1: Fit ComBat on HC ─────────────────────────────────────────────────
combat_params = fit_combat(hc_data, hc_labels)

print("ComBat fitted on HC reference cohort")
print(f"  Scanners:          {combat_params['n_batches']} ({list(combat_params['encoder'].classes_)})")
print(f"  Features:          {combat_params['n_features']}")
print(f"  Reference N:       {combat_params['reference_n_subjects']}")
print(f"  γ̂ shape:           {combat_params['gamma_hat'].shape}  (batch × feature)")
print(f"  Site B additive γ̂ (mean): {combat_params['gamma_hat'][1].mean():.3f}  ← should ≈ {SCANNER_OFFSET:.1f}")

# ── Step A2: Apply blind harmonization to clinical cohorts ───────────────────
adni_h     = apply_combat(adni_data,     clinical_labels, combat_params)
aud_h      = apply_combat(aud_data,      clinical_labels, combat_params)
epilepsy_h = apply_combat(epilepsy_data, clinical_labels, combat_params)

print("\n✓ Blind harmonization applied to all clinical cohorts.")
print(f"  ADNI harmonized:      finite={np.isfinite(adni_h).all()}, shape={adni_h.shape}")
print(f"  AUD harmonized:       finite={np.isfinite(aud_h).all()}, shape={aud_h.shape}")
print(f"  Epilepsy harmonized:  finite={np.isfinite(epilepsy_h).all()}, shape={epilepsy_h.shape}")

In [ ]:
# ── Visualize Module A: Before / After harmonization ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before
axes[0].hist(hc_site_A.mean(axis=0), bins=20, alpha=0.75, color='steelblue',    label='HCP_3T')
axes[0].hist(hc_site_B.mean(axis=0), bins=20, alpha=0.75, color='coral',         label='HCP_7T')
axes[0].set_title('Before Harmonization\n(scanner offset visible)', fontsize=12)
axes[0].set_xlabel('Per-Subject Feature Mean'); axes[0].legend()
axes[0].axvline(hc_site_A.mean(), color='steelblue', ls='--', lw=1.5)
axes[0].axvline(hc_site_B.mean(), color='coral',     ls='--', lw=1.5)

# After
hc_site_B_h = apply_combat(hc_site_B, ['HCP_7T'] * (N_HC // 2), combat_params)
axes[1].hist(hc_site_A.mean(axis=0),    bins=20, alpha=0.75, color='steelblue',      label='HCP_3T (reference)')
axes[1].hist(hc_site_B_h.mean(axis=0), bins=20, alpha=0.75, color='mediumseagreen', label='HCP_7T (harmonized)')
axes[1].set_title('After Blind neuroCombat\n(distributions aligned)', fontsize=12)
axes[1].set_xlabel('Per-Subject Feature Mean'); axes[1].legend()

plt.suptitle('Module A: Site Effect Removal (Blind neuroCombat)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('module_A_harmonization.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Module B-1 — Directed A Matrix via Spectral Inversion

**Dr. Agarwal's challenge addressed:**  
*"MVAR can become computationally unstable on dense parcellations...  
I expect a method that preserves physical validity while ensuring the Gramian does not fail."*

The Spectral Inversion solver guarantees **spectral radius < 1** by construction via Tikhonov-damped eigendecomposition.

In [ ]:
# Derive FC matrix from harmonized HC data (realistic pipeline step)
fc_matrix = np.corrcoef(hc_data)   # (N_PARCELS × N_PARCELS)

# Solve for directed A matrix
A, stability_info = spectral_inversion_solver(fc_matrix, alpha=0.1, system='discrete')

is_stable, sr = check_schur_stability(A)

print("=== Spectral Inversion Solver ===")
print(f"  A matrix shape:      {A.shape}")
print(f"  Spectral radius:     {stability_info['spectral_radius']:.6f}  ← must be < 1.0 ✓")
print(f"  System stable:       {stability_info['is_stable']}")
print(f"  Condition number:    {stability_info['condition_number']:.2f}")
print(f"  A is asymmetric:     {not np.allclose(A, A.T)}  (directed connectivity)")
print(f"  NaN/Inf free:        {np.isfinite(A).all()}")

In [ ]:
# Normalize A for downstream Gramian / energy computation
A_norm = normalize_matrix(A, system='discrete')
is_norm_stable, sr_norm = check_schur_stability(A_norm)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vmax = max(np.abs(A).max(), np.abs(A_norm).max())
for ax, mat, title in zip(axes, [A, A_norm], [f'Raw A\n(sr={sr:.4f})', f'Normalized A\n(sr={sr_norm:.4f})']):
    im = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Source Node'); ax.set_ylabel('Target Node')
    plt.colorbar(im, ax=ax, label='Weight')

plt.suptitle('Module B-1: Directed A Matrix (Spectral Inversion)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('module_B1_a_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Module B-2 — Controllability Gramian & Minimum Energy

The **Gramian** encodes which states are reachable with finite energy.  
**Minimum Energy** quantifies the cost of transitioning between specific brain states — a direct proxy for neurological state change difficulty.

In [ ]:
T_HORIZON = 5       # control horizon (time steps)
B = np.eye(N_PARCELS)  # full control matrix (all nodes are controller nodes)

# Compute Controllability Gramian
Wc = compute_gramian(A_norm, T=T_HORIZON, B=B, system='discrete')

eigvals_Wc = np.linalg.eigvalsh(Wc)

print("=== Controllability Gramian ===")
print(f"  Shape:           {Wc.shape}")
print(f"  Symmetric:       {np.allclose(Wc, Wc.T, atol=1e-8)}")
print(f"  Min eigenvalue:  {eigvals_Wc.min():.6f}  (≥ 0 → PSD ✓)")
print(f"  Max eigenvalue:  {eigvals_Wc.max():.4f}")
print(f"  Rank (approx):   {np.sum(eigvals_Wc > 1e-10)} / {N_PARCELS}")

In [ ]:
# Define 4 clinically meaningful brain states
x_healthy   = np.zeros(N_PARCELS); x_healthy[:N_PARCELS//4]          = 1.0   # HC baseline
x_adni      = np.zeros(N_PARCELS); x_adni[N_PARCELS//4:N_PARCELS//2] = 1.0   # AD state
x_aud       = np.zeros(N_PARCELS); x_aud[N_PARCELS//2:3*N_PARCELS//4]= 1.0   # AUD state
x_epilepsy  = np.zeros(N_PARCELS); x_epilepsy[3*N_PARCELS//4:]        = 1.0   # Epilepsy state

states = {
    'Healthy':  x_healthy,
    'ADNI-AD':  x_adni,
    'AUD':      x_aud,
    'Epilepsy': x_epilepsy,
}

state_names = list(states.keys())
n_states = len(state_names)

# Compute pairwise transition energies
energy_matrix = np.zeros((n_states, n_states))
for i, si in enumerate(state_names):
    for j, sj in enumerate(state_names):
        E = minimum_energy(A_norm, T=T_HORIZON, B=B,
                           x0=states[si], xf=states[sj],
                           system='discrete')
        energy_matrix[i, j] = E.sum()

np.fill_diagonal(energy_matrix, 0)    # zero cost for self-transitions

print("Pairwise transition energy matrix (x0 → xf):")
print(np.array2string(energy_matrix, formatter={'float_kind': lambda x: f"{x:8.4f}"}))
print()
print(f"  Healthy → ADNI-AD:   {energy_matrix[0,1]:.4f}")
print(f"  Healthy → Epilepsy:  {energy_matrix[0,3]:.4f}  (seizure transition cost)")
print(f"  Healthy → AUD:       {energy_matrix[0,2]:.4f}  (addiction reinforcement)")
print(f"  Epilepsy → Healthy:  {energy_matrix[3,0]:.4f}  (restorative effort)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Energy heatmap
sns.heatmap(
    energy_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
    xticklabels=state_names, yticklabels=state_names, ax=axes[0],
    linewidths=0.5, linecolor='white', cbar_kws={'label': 'Control Energy'}
)
axes[0].set_title('Pairwise State Transition Energy\n(x₀ → x_f)', fontsize=12)
axes[0].set_xlabel('Target State (xf)')
axes[0].set_ylabel('Initial State (x₀)')

# Gramian eigenspectrum
axes[1].semilogy(sorted(eigvals_Wc, reverse=True), 'o-', color='steelblue', ms=4, lw=1.5)
axes[1].set_title('Gramian Eigenspectrum\n(energy landscape richness)', fontsize=12)
axes[1].set_xlabel('Eigenvalue rank'); axes[1].set_ylabel('Eigenvalue (log scale)')
axes[1].axhline(1e-10, color='red', ls='--', lw=1, label='PSD threshold')
axes[1].legend()

plt.suptitle('Module B-2: Minimum Control Energy', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('module_B2_energy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Module C — Facilitator Node Detection

**Facilitator nodes** are brain regions that can drive the network into *hard-to-reach disease states*  
with low energy input — these are the anatomical **seizure gateways** and **AUD craving circuit nodes**  
that are primary neurostimulation targets.

**Modal Controllability** (high value = node can reach difficult states)  
**Average Controllability** (high value = node has broad influence over many states)

In [ ]:
mc = modal_controllability(A_norm)
ac = average_controllability(A_norm)

top_nodes, top_scores = rank_facilitator_nodes(A_norm, top_k=10)

print(f"Modal Controllability range:   [{mc.min():.4f}, {mc.max():.4f}]")
print(f"Average Controllability range: [{ac.min():.4f}, {ac.max():.4f}]")
print()
print("Top 10 Facilitator Nodes (by Modal Controllability):")
print("-" * 55)
for rank, (node, score) in enumerate(zip(top_nodes, top_scores), 1):
    bar = '█' * int(score / top_scores[0] * 25)
    print(f"  Rank {rank:2d} | Node {node:3d} | MC = {score:.5f}  {bar}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# ── Panel 1: Modal controllability bar ────────────────────────────────────────
ax = axes[0, 0]
colors_mc = plt.cm.Reds(mc / mc.max())
ax.bar(range(N_PARCELS), mc, color=colors_mc, edgecolor='none', width=1.0)
ax.bar(top_nodes, mc[top_nodes], color='darkred', width=1.0, label='Top 10 Facilitators')
ax.set_title('Modal Controllability\n(Seizure / AUD gateway nodes)', fontsize=11)
ax.set_xlabel('Node Index'); ax.set_ylabel('MC Score')
ax.legend(fontsize=9)

# ── Panel 2: Average controllability bar ──────────────────────────────────────
ax = axes[0, 1]
colors_ac = plt.cm.Blues(ac / ac.max())
ax.bar(range(N_PARCELS), ac, color=colors_ac, edgecolor='none', width=1.0)
ax.set_title('Average Controllability\n(Broad stimulation targets)', fontsize=11)
ax.set_xlabel('Node Index'); ax.set_ylabel('AC Score')

# ── Panel 3: MC vs AC scatter — role separation ───────────────────────────────
ax = axes[1, 0]
sc = ax.scatter(ac, mc, c=mc, cmap='RdYlGn_r', s=40, alpha=0.8, edgecolors='none')
ax.scatter(ac[top_nodes], mc[top_nodes], s=120, facecolors='none', edgecolors='darkred',
           linewidths=2, label='Top 10 Facilitators', zorder=5)
for n in top_nodes[:5]:
    ax.annotate(f'N{n}', (ac[n], mc[n]), fontsize=7, ha='left',
                xytext=(3, 3), textcoords='offset points')
plt.colorbar(sc, ax=ax, label='MC Score')
ax.set_xlabel('Average Controllability')
ax.set_ylabel('Modal Controllability')
ax.set_title('MC vs AC — Node Role Separation\n(Facilitators: high MC, moderate AC)', fontsize=11)
ax.legend(fontsize=9)

# ── Panel 4: PCA of energy profiles with clinical cohort labels ───────────────
n_per_group = 20
states_list = [
    ('Healthy',  x_healthy,  'steelblue',  'o'),
    ('ADNI-AD',  x_adni,     'coral',      's'),
    ('AUD',      x_aud,      'mediumpurple','D'),
    ('Epilepsy', x_epilepsy, 'seagreen',   '^'),
]

all_energies = []
all_labels   = []
for name, x_base, color, marker in states_list:
    for _ in range(n_per_group):
        x0 = x_base + RNG.normal(0, 0.1, N_PARCELS)
        xf = x_healthy + RNG.normal(0, 0.05, N_PARCELS)
        E  = minimum_energy(A_norm, T=T_HORIZON, B=B, x0=x0, xf=xf, system='discrete')
        all_energies.append(E)
    all_labels.extend([name] * n_per_group)

all_energies = np.array(all_energies)
pca = PCA(n_components=2, random_state=42)
embedding = pca.fit_transform(all_energies)

ax = axes[1, 1]
for name, x_base, color, marker in states_list:
    mask = [l == name for l in all_labels]
    ax.scatter(embedding[mask, 0], embedding[mask, 1],
               c=color, marker=marker, s=70, alpha=0.8,
               edgecolors='white', linewidths=0.4, label=name)

ax.set_title(
    f'PCA: Clinical State Space (Energy Profiles)\n'
    f'PC1: {pca.explained_variance_ratio_[0]*100:.1f}% | '
    f'PC2: {pca.explained_variance_ratio_[1]*100:.1f}%',
    fontsize=11
)
ax.set_xlabel(f'PC1')
ax.set_ylabel(f'PC2')
ax.legend(title='Cohort', fontsize=9)

plt.suptitle('Module C: Facilitator Nodes & Clinical State Space', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('module_C_facilitators.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Pipeline Summary

| Module | Method | Key Result |
|--------|--------|------------|
| **A: Harmonization** | Blind neuroCombat | Scanner bias removed; disease biomarkers preserved |
| **B-1: Connectivity** | Spectral Inversion | Schur-stable directed A matrix (sr < 1.0) |
| **B-2: Control Engine** | Gramian + Min. Energy | Pairwise state transition cost quantified |
| **C: Facilitator Nodes** | Modal Controllability | Top-k gateway nodes ranked and visualized |

### Validation Checklist

- ✅ ComBat fitted **only** on HC — blind guarantee maintained  
- ✅ Spectral radius < 1.0 — Gramian computation is numerically stable  
- ✅ Gramian is PSD (all eigenvalues ≥ 0)  
- ✅ Facilitator nodes identified (top 10 by MC score)  
- ✅ All outputs are NaN / Inf free  
- ✅ Full pipeline reproducible with fixed seed 2026  

### Next Steps (GSoC Implementation)

1. Replace synthetic data with real BIDS datasets (HCP, ADNI, OpenNeuro AUD/Epilepsy)  
2. Map facilitator node scores onto cortical surface via `nilearn.plotting.plot_surf_stat_map`  
3. Add BIDS-App CLI wrapper for end-to-end automation  
4. Validate disease cohort separability (silhouette score on energy profile PCA)  
5. Export harmonized features to `derivatives/neurosim/` in BIDS-compatible format  